# Tarea para el Hogar 02

Esta Tarea para el Hogar 02 se entrega el final de la segunda clase
<br> se espera de usted que intente avanzar con los desafios propuestos y que los traiga terminados para la Clase 03, ya que se analizarán los resultados

##  1. Ensembles de Modelos

Vea el siguiente video [BBC - The Code - The Wisdom of the Crowd](https://www.youtube.com/watch?v=iOucwX7Z1HU)    ( 5 min)


Lea los siguientes artículos


*   [The Wisdom of Crowds (Vox Populi) by Francis Galton](https://www.all-about-psychology.com/the-wisdom-of-crowds.html)  (10 min)
*   [A Gentle Introduction to Ensemble Learning](https://machinelearningmastery.com/what-is-ensemble-learning/)  (10 min)





---



##  2.  Zero2Hero   primera parte
Se han lanzado los primeros fascículos coleccionables llamados "from Zero to Hero" que muy detalladamente, paso a paso enseñan todo lo necesario de R para entender los scripts oficiales de la asignatura.
Están en el repositorio oficial de la asignatura, carpeta  **src/zero2hero**



---



## 3.  Grid Search

Busque en internet el precido significado de los hiperparámetros de la librería **rpart**  que está implementando el algoritmo **CART**  Classification and Regression Trees  propuesto en el año 1984 por Leo Brieman:

*   cp
*   maxdepth
*   minsplit
*   minbucket

Entienda que valores es razonable tome cada hiperparámetro,  en particular profundice en el hiperparámetro  **cp**  y la posibilidad que tome valores negativos.  Es válido consultar a su amigo de *capacidades especiales*  ChatGPT


En las siguientes celdas a un notebook incompleto, un esqueleto de codigo brindado a modo de facilitarle la tarea de codeo y permitir que su valiosa cognición se concentre temas conceptuales de Ciencia de Datos

Modifiquelo agregando loops para que recorra TODOS los hiperparámetros de rpart  < cp, maxdepth, minsplit, minbucket >, y luego póngalo a correr. Recuerde cambiar por SU semilla
Tenga muy presente la granularidad que eligirá para cada hiperparámetro.

### Seteo del ambiente en Google Colab

Esta parte se debe correr con el runtime en Python3
<br>Ir al menu, Runtime -> Change Runtime Tipe -> Runtime type ->  **Python 3**

Conectar la virtual machine donde esta corriendo Google Colab con el  Google Drive, para poder tener persistencia de archivos

In [1]:
# primero establecer el Runtime de Python 3
from google.colab import drive
drive.mount('/content/.drive')

Mounted at /content/.drive


Para correr la siguiente celda es fundamental en Arranque en Frio haber copiado el archivo kaggle.json al Google Drive, en la carpeta indicada en el instructivo

<br>los siguientes comando estan en shell script de Linux
*   Crear las carpetas en el Google Drive
*   "instalar" el archivo kaggle.json desde el Google Drive a la virtual machine para que pueda ser utilizado por la libreria  kaggle de Python
*   Bajar el  **dataset_pequeno**  al  Google Drive  y tambien al disco local de la virtual machine que esta corriendo Google Colab



In [2]:
%%shell

mkdir -p "/content/.drive/My Drive/dmeyf"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/dmeyf"   /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json


mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets


# defino funcion descargar()
descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/utn2026-b40a/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}


# hago la descarga efectiva, llamando a descargar()
descargar  "dataset_pequeno.csv"


limpio el ambiente de R

In [1]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,668153,35.7,1473300,78.7,1425955,76.2
Vcells,1236310,9.5,8388608,64.0,1978711,15.1


In [2]:
# cargo las librerias que necesito
require("data.table")
require("rpart")
require("parallel")
if (!require("primes")) install.packages("primes")
require("primes")

Loading required package: data.table


Attaching package: ‘data.table’


The following object is masked from ‘package:base’:

    %notin%


Loading required package: rpart

Loading required package: parallel

Loading required package: primes

Warning message in library(package, lib.loc = lib.loc, character.only = TRUE, logical.return = TRUE, :
“there is no package called ‘primes’”
Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Loading required package: primes



Aqui debe poner SU semiila primigenia

In [3]:
PARAM <- list()
# reemplazar por su primer semilla
PARAM$semilla_primigenia <- 346321
PARAM$qsemillas <- 1

PARAM$training_pct <- 70L  # entre  1L y 99L

# elegir SU dataset comentando/ descomentando
PARAM$dataset_nom <- "~/datasets/dataset_pequeno.csv"

In [4]:
# particionar agrega una columna llamada fold a un dataset
#  que consiste en una particion estratificada segun agrupa
# particionar( data=dataset, division=c(70,30), agrupa=clase_ternaria, seed=semilla)
#   crea una particion 70, 30

particionar <- function(data, division, agrupa = "", campo = "fold", start = 1, seed = NA) {
  if (!is.na(seed)) set.seed(seed)

  bloque <- unlist(mapply(function(x, y) {
    rep(y, x)
  }, division, seq(from = start, length.out = length(division))))

  data[, (campo) := sample(rep(bloque, ceiling(.N / length(bloque))))[1:.N],
    by = agrupa
  ]
}


In [5]:
ArbolEstimarGanancia <- function(semilla, training_pct, param_basicos) {
  # particiono estratificadamente el dataset
  particionar(dataset,
    division = c(training_pct, 100L -training_pct),
    agrupa = "clase_ternaria",
    seed = semilla # aqui se usa SU semilla
  )

  # genero el modelo
  # predecir clase_ternaria a partir del resto
  modelo <- rpart("clase_ternaria ~ .",
    data = dataset[fold == 1], # fold==1  es training,  el 70% de los datos
    xval = 0,
    control = param_basicos
  ) # aqui van los parametros del arbol

  # aplico el modelo a los datos de testing
  prediccion <- predict(modelo, # el modelo que genere recien
    dataset[fold == 2], # fold==2  es testing, el 30% de los datos
    type = "prob"
  ) # type= "prob"  es que devuelva la probabilidad

  # prediccion es una matriz con TRES columnas,
  #  llamadas "BAJA+1", "BAJA+2"  y "CONTINUA"
  # cada columna es el vector de probabilidades


  # calculo la ganancia en testing  qu es fold==2
  ganancia_test <- dataset[
    fold == 2,
    sum(ifelse(prediccion[, "BAJA+2"] > 0.025,
      ifelse(clase_ternaria == "BAJA+2", 975000, -25000),
      0
    ))
  ]

  # escalo la ganancia como si fuera todo el dataset
  ganancia_test_normalizada <- ganancia_test / (( 100 - PARAM$training_pct ) / 100 )

  return(
    c( list("semilla" = semilla),
      param_basicos,
      list( "ganancia_test" = ganancia_test_normalizada )
     )
  )
}


In [6]:
ArbolesMontecarlo <- function(semillas, param_basicos) {

  # la funcion mcmapply  llama a la funcion ArbolEstimarGanancia
  #  tantas veces como valores tenga el vector  PARAM$semillas
  salida <- mcmapply(ArbolEstimarGanancia,
    semillas, # paso el vector de semillas
    MoreArgs = list(PARAM$training_pct, param_basicos), # aqui paso el segundo parametro
    SIMPLIFY = FALSE,
    mc.cores = detectCores()
  )

  return(salida)
}


In [7]:
# carpeta de trabajo
# por favor cambiar numero de experimento si se cambia el loop principal
setwd("/content/buckets/b1/exp")
experimento <- "HT333"
dir.create(experimento, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento ))

In [8]:
# lectura del dataset
dataset <- fread("/content/datasets/dataset_pequeno.csv")

# trabajo solo con los datos con clase, es decir 202107
dataset <- dataset[clase_ternaria != ""]

In [9]:

# genero numeros primos
primos <- generate_primes(min = 100000, max = 1000000)
set.seed(PARAM$semilla_primigenia) # inicializo
# me quedo con PARAM$qsemillas   semillas
PARAM$semillas <- sample(primos, PARAM$qsemillas )


In [10]:
# genero la data.table donde van los resultados detallados del Grid Search
# un registro para cada combinacion de < semilla, parametros >

if(file.exists("gridsearch_detalle.txt")){
  tb_grid_search_detalle <- fread("gridsearch_detalle.txt")
}else{
  tb_grid_search_detalle <- data.table(
    semilla = integer(),
    cp = numeric(),
    maxdepth = integer(),
    minsplit = integer(),
    minbucket = integer(),
    ganancia_test = numeric()
  )
}

nrow( tb_grid_search_detalle )

[1] 0

Esta es la parte del código que usted debe expandir a TODOS los hiperparámetros de rpart,
<br>ya que actualmente apenas recorre  maxdepth y  minsplit  dejando fijos  cp=-0.5  y minbucket=5

In [12]:

# itero por los loops anidados para cada hiperparametro
iter <- 0

for (vmax_depth in c(12)) {
  for (vmin_split in c(200)) {
    # notar como se agrega

    iter <- iter + 1
    cat( iter, " " )
    flush.console()
    if( iter*PARAM$qsemillas < nrow(tb_grid_search_detalle)+1 ) next

    # vminsplit  minima cantidad de registros en un nodo para hacer el split
    param_basicos <- list(
      "cp" = -1, # complejidad minima
      "maxdepth" = vmax_depth, # profundidad máxima del arbol
      "minsplit" = vmin_split, # tamaño minimo de nodo para hacer split
      "minbucket" = 66 # minima cantidad de registros en una hoja
    )

    # Un solo llamado, con la semilla 17
    ganancias <- ArbolesMontecarlo(PARAM$semillas, param_basicos)

    # agrego a la tabla
    tb_grid_search_detalle <- rbindlist(
      list( tb_grid_search_detalle,
            rbindlist(ganancias) )
    )

  }

  # grabo cada vez TODA la tabla en el loop mas externo
  fwrite( tb_grid_search_detalle,
          file = "gridsearch_detalle.txt",
          sep = "\t" )
}


1  

### Etapa 0: diagnóstico rápido — ¿el `maxdepth` óptimo depende de `cp` ?
Antes de fijar los vectores del grid completo, corremos una grilla chica (54 combinaciones, ~1.25hs estimadas) cruzando 3 valores representativos de `cp` con las 6 profundidades y 3 valores representativos de `minsplit` ( `minbucket = 5` fijo, igual que en el loop simple ).

El objetivo NO es encontrar el óptimo definitivo, sino chequear si el `maxdepth = 6` que ganó en el loop simple (con `cp = -0.5` fijo) se sostiene para otros valores de `cp`, o si el óptimo se mueve. Esto determina si conviene angostar `maxdepth` en el grid completo o dejarlo con todos sus valores.

Usa su propio archivo de checkpoint ( `gridsearch_detalle_diagnostico.txt` ) para no mezclarse con las otras corridas.

In [ ]:

# Etapa 0: diagnostico rapido, ver si el maxdepth ganador depende de cp
#  minbucket queda fijo en 5 (igual que en el loop simple)
#  minsplit se reduce a 3 valores representativos, ya que la corrida anterior
#   mostro que entre 400 y 1000 la ganancia empataba (no hace falta mas resolucion ahi)

Sys.time()

archivo_grid_diagnostico <- "gridsearch_detalle_diagnostico.txt"

if (file.exists(archivo_grid_diagnostico)) {
  tb_grid_search_detalle <- fread(archivo_grid_diagnostico)
} else {
  tb_grid_search_detalle <- data.table(
    semilla = integer(),
    cp = numeric(),
    maxdepth = integer(),
    minsplit = integer(),
    minbucket = integer(),
    ganancia_test = numeric()
  )
}

iter <- 0

for (vcp in c(-1, -0.5, 0.01)) {
  for (vmax_depth in c(4, 6, 8, 10, 12, 14)) {
    for (vmin_split in c(1000, 200, 20)) {

      iter <- iter + 1
      cat( iter, " " )
      flush.console()
      if( iter*PARAM$qsemillas < nrow(tb_grid_search_detalle)+1 ) next

      param_basicos <- list(
        "cp" = vcp,
        "maxdepth" = vmax_depth,
        "minsplit" = vmin_split,
        "minbucket" = 5 # fijo, igual que en el loop simple
      )

      ganancias <- ArbolesMontecarlo(PARAM$semillas, param_basicos)

      tb_grid_search_detalle <- rbindlist(
        list( tb_grid_search_detalle,
              rbindlist(ganancias) )
      )
    }
  }

  # grabo cada vez que termino un valor de cp
  fwrite( tb_grid_search_detalle,
          file = archivo_grid_diagnostico,
          sep = "\t" )
}

Sys.time()

[1] "2026-08-15 02:40:19 UTC"

1  2  3  4  5  6  7  8  9  10  11  12  13  14  15  16  17  18  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35  36  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53  54  

[1] "2026-08-15 03:32:57 UTC"

In [13]:
# analisis del diagnostico: mejor maxdepth para cada valor de cp
# si la columna maxdepth se mantiene estable entre filas, el optimo NO depende de cp
# y se puede angostar el vector de maxdepth en el grid completo con confianza

tb_diag_resumen <- tb_grid_search_detalle[,
  list( ganancia_mean = mean(ganancia_test) ),
  list( cp, maxdepth )
]
setorder( tb_diag_resumen, cp, -ganancia_mean )

# mejor maxdepth por cada cp
tb_diag_resumen[, .SD[1], by = cp]


cp,maxdepth,ganancia_mean
<dbl>,<dbl>,<dbl>
-1,12,432833333


### Etapa 0-bis: cubrir la zona intermedia de `cp` que faltó
La etapa anterior mostró dos cosas: `cp=-1` y `cp=-0.5` dieron resultados **idénticos** (ambos desactivan por completo la poda por complejidad), y `cp=0.01` dio **ganancia 0 en todas las combinaciones** (el árbol queda sin splits). En la práctica solo cubrimos 2 regímenes distintos de `cp` (sin poda / poda total), no 3.

Falta la zona intermedia: `cp` positivo pero chico (`0.001`, `0.005`), donde el árbol probablemente sí poda algo sin quedar muerto. Ahí es donde más chances hay de que el `maxdepth` óptimo sea distinto. Corremos 12 combinaciones (`minsplit=1000` fijo, ya que es el representante de la franja ganadora) para chequearlo barato.

Usa su propio checkpoint ( `gridsearch_detalle_diagnostico2.txt` ) para no mezclarse con las otras corridas.

In [ ]:

# Etapa 0-bis: cubrir cp positivo chico (zona intermedia), minsplit fijo en 1000
#  minbucket sigue fijo en 5, igual que en las etapas anteriores

archivo_grid_diagnostico2 <- "gridsearch_detalle_diagnostico2.txt"

if (file.exists(archivo_grid_diagnostico2)) {
  tb_grid_search_detalle <- fread(archivo_grid_diagnostico2)
} else {
  tb_grid_search_detalle <- data.table(
    semilla = integer(),
    cp = numeric(),
    maxdepth = integer(),
    minsplit = integer(),
    minbucket = integer(),
    ganancia_test = numeric()
  )
}

iter <- 0

for (vcp in c(0.001, 0.005)) {
  for (vmax_depth in c(4, 6, 8, 10, 12, 14)) {

    iter <- iter + 1
    cat( iter, " " )
    flush.console()
    if( iter*PARAM$qsemillas < nrow(tb_grid_search_detalle)+1 ) next

    param_basicos <- list(
      "cp" = vcp,
      "maxdepth" = vmax_depth,
      "minsplit" = 1000,
      "minbucket" = 5
    )

    ganancias <- ArbolesMontecarlo(PARAM$semillas, param_basicos)

    tb_grid_search_detalle <- rbindlist(
      list( tb_grid_search_detalle,
            rbindlist(ganancias) )
    )
  }

  # grabo cada vez que termino un valor de cp
  fwrite( tb_grid_search_detalle,
          file = archivo_grid_diagnostico2,
          sep = "\t" )
}


1  2  3  4  5  6  7  8  9  10  11  12  

### Etapa 0-ter: el caso límite `cp = 0`
Con `cp` negativo el árbol funciona bien, y con `cp` positivo (por más chico que sea: `0.001`, `0.005`, `0.01`) el árbol queda muerto (`ganancia_test = 0`). Falta el punto límite exacto: `cp = 0`. Corremos 6 combinaciones (`minsplit = 1000` fijo) para ver de qué lado cae.

Usa su propio checkpoint ( `gridsearch_detalle_diagnostico3.txt` ) para no mezclarse con las otras corridas.

In [ ]:

# Etapa 0-ter: caso limite cp = 0, minsplit fijo en 1000, minbucket fijo en 5

archivo_grid_diagnostico3 <- "gridsearch_detalle_diagnostico3.txt"

if (file.exists(archivo_grid_diagnostico3)) {
  tb_grid_search_detalle <- fread(archivo_grid_diagnostico3)
} else {
  tb_grid_search_detalle <- data.table(
    semilla = integer(),
    cp = numeric(),
    maxdepth = integer(),
    minsplit = integer(),
    minbucket = integer(),
    ganancia_test = numeric()
  )
}

iter <- 0

for (vmax_depth in c(4, 6, 8, 10, 12, 14)) {

  iter <- iter + 1
  cat( iter, " " )
  flush.console()
  if( iter*PARAM$qsemillas < nrow(tb_grid_search_detalle)+1 ) next

  param_basicos <- list(
    "cp" = 0,
    "maxdepth" = vmax_depth,
    "minsplit" = 1000,
    "minbucket" = 5
  )

  ganancias <- ArbolesMontecarlo(PARAM$semillas, param_basicos)

  tb_grid_search_detalle <- rbindlist(
    list( tb_grid_search_detalle,
          rbindlist(ganancias) )
  )
}

fwrite( tb_grid_search_detalle,
        file = archivo_grid_diagnostico3,
        sep = "\t" )


1  2  3  4  5  6  

### Loop expandido: recorre los CUATRO hiperparámetros ( cp, maxdepth, minsplit, minbucket )
Version completa del loop de la celda anterior, agregando los loops anidados para  **cp**  y  **minbucket** .
<br>Usa su propio archivo de checkpoint ( `gridsearch_detalle_completo.txt` ) para no mezclarse con la corrida parcial de la celda anterior.

In [ ]:

# itero por los loops anidados para CADA hiperparametro de rpart:  cp, maxdepth, minsplit, minbucket

# vectores ajustados con evidencia de las etapas de diagnostico (0, 0-bis, 0-ter):
#  - cp: de 7 a 2 valores. -1 es identico a -0.5 (ambos desactivan la poda por complejidad).
#    0.001/0.005/0.01 dan ganancia_test=0 siempre (arbol sin splits, la clase minoritaria
#    es demasiado rara para justificar cualquier cp positivo). 0 es un regimen propio,
#    funcional pero distinto de los negativos.
#  - minsplit: de 9 a 6 valores. 1000/800/600/400 empataban exacto (no hay diferencia
#    hasta bajar de 400), asi que dejo un solo representante de esa franja (1000).
#  - minbucket: de 6 a 5 valores. Saco el extremo minbucket=1 (mas propenso a overfitting).
#  - maxdepth: sin cambios, sigue siendo el segundo hiperparametro mas importante.

# uso un archivo de checkpoint propio para no mezclar esta corrida (4 hiperparametros)
#  con la de la celda anterior (2 hiperparametros), que tiene otra numeracion de iter

Sys.time()

archivo_grid_completo <- "gridsearch_detalle_completo.txt"

if (file.exists(archivo_grid_completo)) {
  tb_grid_search_detalle <- fread(archivo_grid_completo)
} else {
  tb_grid_search_detalle <- data.table(
    semilla = integer(),
    cp = numeric(),
    maxdepth = integer(),
    minsplit = integer(),
    minbucket = integer(),
    ganancia_test = numeric()
  )
}

iter <- 0

for (vcp in c(-1, 0)) {
  for (vmax_depth in c(4, 6, 8, 10, 12, 14)) {
    for (vmin_split in c(1000, 200, 100, 50, 20, 10)) {
      for (vmin_bucket in c(5, 10, 20, 50, 100)) {
        # minbucket no puede superar minsplit, es una combinacion redundante
        #  ( ambas hojas resultantes de un split necesitarian minbucket casos,
        #    lo que ya excede minsplit )  la salteo ANTES de incrementar iter
        #  para no desalinear el checkpoint de mas abajo
        if( vmin_bucket > vmin_split ) next

        # notar como se agrega

        iter <- iter + 1
        cat( iter, " " )
        flush.console()
        if( iter*PARAM$qsemillas < nrow(tb_grid_search_detalle)+1 ) next

        param_basicos <- list(
          "cp" = vcp, # complejidad minima, puede ser negativaMe
          "maxdepth" = vmax_depth, # profundidad máxima del arbol
          "minsplit" = vmin_split, # tamaño minimo de nodo para hacer split
          "minbucket" = vmin_bucket # minima cantidad de registros en una hoja
        )

        # Un llamado por cada combinacion, recorriendo TODAS las semillas
        ganancias <- ArbolesMontecarlo(PARAM$semillas, param_basicos)

        # agrego a la tabla
        tb_grid_search_detalle <- rbindlist(
          list( tb_grid_search_detalle,
                rbindlist(ganancias) )
        )

      }
    }
  }

  # grabo cada vez TODA la tabla, al terminar cada valor de cp (loop mas externo)
  fwrite( tb_grid_search_detalle,
          file = archivo_grid_completo,
          sep = "\t" )
}

Sys.time()

[1] "2026-08-15 11:07:55 UTC"

1  2  3  4  5  6  7  8  9  10  11  12  13  14  15  16  17  18  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35  36  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53  54  55  56  57  58  59  60  61  62  63  64  65  66  67  68  69  70  71  72  73  74  75  76  77  78  79  80  81  82  83  84  85  86  87  88  89  90  91  92  93  94  95  96  97  98  99  100  101  102  103  104  105  106  107  108  109  110  111  112  113  114  115  116  117  118  119  120  121  122  123  124  125  126  127  128  129  130  131  132  133  134  135  136  137  138  139  140  141  142  143  144  145  146  147  148  149  150  151  152  153  154  155  156  157  158  159  160  161  162  163  164  165  166  167  168  169  170  171  172  173  174  175  176  177  178  179  180  181  182  183  184  185  186  187  188  189  190  191  192  193  194  195  196  197  198  199  200  201  202  203  204  205  206  207  208  209  210  211  212  213  214  215  216  217  218  219  220  221  222

[1] "2026-08-15 17:14:41 UTC"

### Sintonía fina de los parámetros

In [ ]:
# itero por los loops anidados para CADA hiperparametro de rpart:  cp, maxdepth, minsplit, minbucket


Sys.time()

archivo_grid_completo <- "gridsearch_detalle_completo2.txt"

if (file.exists(archivo_grid_completo)) {
  tb_grid_search_detalle <- fread(archivo_grid_completo)
} else {
  tb_grid_search_detalle <- data.table(
    semilla = integer(),
    cp = numeric(),
    maxdepth = integer(),
    minsplit = integer(),
    minbucket = integer(),
    ganancia_test = numeric()
  )
}

iter <- 0

for (vcp in c(-1)) {
  for (vmax_depth in c(6, 8)) {
    for (vmin_split in c(20, 50, 10)) {
      for (vmin_bucket in c(5, 10, 20, 50, 100)) {
        # minbucket no puede superar minsplit, es una combinacion redundante
        #  ( ambas hojas resultantes de un split necesitarian minbucket casos,
        #    lo que ya excede minsplit )  la salteo ANTES de incrementar iter
        #  para no desalinear el checkpoint de mas abajo
        if( vmin_bucket > vmin_split ) next

        # notar como se agrega

        iter <- iter + 1
        cat( iter, " " )
        flush.console()
        if( iter*PARAM$qsemillas < nrow(tb_grid_search_detalle)+1 ) next

        param_basicos <- list(
          "cp" = vcp, # complejidad minima, puede ser negativaMe
          "maxdepth" = vmax_depth, # profundidad máxima del arbol
          "minsplit" = vmin_split, # tamaño minimo de nodo para hacer split
          "minbucket" = vmin_bucket # minima cantidad de registros en una hoja
        )

        # Un llamado por cada combinacion, recorriendo TODAS las semillas
        ganancias <- ArbolesMontecarlo(PARAM$semillas, param_basicos)

        # agrego a la tabla
        tb_grid_search_detalle <- rbindlist(
          list( tb_grid_search_detalle,
                rbindlist(ganancias) )
        )

      }
    }
  }

  # grabo cada vez TODA la tabla, al terminar cada valor de cp (loop mas externo)
  fwrite( tb_grid_search_detalle,
          file = archivo_grid_completo,
          sep = "\t" )
}

Sys.time()

[1] "2026-08-18 05:36:16 UTC"

1  2  3  4  5  6  7  8  9  10  11  12  13  14  15  16  17  18  

[1] "2026-08-18 05:49:43 UTC"

### Agregamos 6 semillas para las posibles combinaciones optimas

In [ ]:
# itero por los loops anidados para CADA hiperparametro de rpart:  cp, maxdepth, minsplit, minbucket


Sys.time()

archivo_grid_completo <- "gridsearch_detalle_completo5.txt"

if (file.exists(archivo_grid_completo)) {
  tb_grid_search_detalle <- fread(archivo_grid_completo)
} else {
  tb_grid_search_detalle <- data.table(
    semilla = integer(),
    cp = numeric(),
    maxdepth = integer(),
    minsplit = integer(),
    minbucket = integer(),
    ganancia_test = numeric()
  )
}

iter <- 0

for (vcp in c(-1)) {
  for (vmax_depth in c(8)) {
    for (vmin_split in c(1000)) {
      for (vmin_bucket in c(75, 100, 125, 150)) {
        # minbucket no puede superar minsplit, es una combinacion redundante
        #  ( ambas hojas resultantes de un split necesitarian minbucket casos,
        #    lo que ya excede minsplit )  la salteo ANTES de incrementar iter
        #  para no desalinear el checkpoint de mas abajo
        if( vmin_bucket > vmin_split ) next

        # notar como se agrega

        iter <- iter + 1
        cat( iter, " " )
        flush.console()
        if( iter*PARAM$qsemillas < nrow(tb_grid_search_detalle)+1 ) next

        param_basicos <- list(
          "cp" = vcp, # complejidad minima, puede ser negativaMe
          "maxdepth" = vmax_depth, # profundidad máxima del arbol
          "minsplit" = vmin_split, # tamaño minimo de nodo para hacer split
          "minbucket" = vmin_bucket # minima cantidad de registros en una hoja
        )

        # Un llamado por cada combinacion, recorriendo TODAS las semillas
        ganancias <- ArbolesMontecarlo(PARAM$semillas, param_basicos)

        # agrego a la tabla
        tb_grid_search_detalle <- rbindlist(
          list( tb_grid_search_detalle,
                rbindlist(ganancias) )
        )

      }
    }
  }

  # grabo cada vez TODA la tabla, al terminar cada valor de cp (loop mas externo)
  fwrite( tb_grid_search_detalle,
          file = archivo_grid_completo,
          sep = "\t" )
}

Sys.time()

[1] "2026-08-18 08:59:43 UTC"

1  2  3  4  

[1] "2026-08-18 09:15:16 UTC"

In [ ]:
fwrite( tb_grid_search_detalle,
   file = "gridsearch_detalle5.txt",
   sep = "\t"
)

In [ ]:
# cantidad de registros de la tabla
nrow(tb_grid_search_detalle)

[1] 20

In [ ]:
# muestro la tabla
tb_grid_search_detalle

semilla,cp,maxdepth,minsplit,minbucket,ganancia_test
<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
187211,-1,8,1000,75,501500000
430267,-1,8,1000,75,507416667
322247,-1,8,1000,75,469833333
906839,-1,8,1000,75,460333333
569321,-1,8,1000,75,481916667
187211,-1,8,1000,100,509083333
430267,-1,8,1000,100,488083333
322247,-1,8,1000,100,485666667
906839,-1,8,1000,100,456916667


In [ ]:
# genero y grabo el resumen
tb_grid_search <- tb_grid_search_detalle[,
  list( "ganancia_mean" = mean(ganancia_test),
    "qty" = .N ),
  list( cp, maxdepth, minsplit, minbucket )
]


In [ ]:
# ordeno descendente por ganancia
setorder( tb_grid_search, -ganancia_mean )


In [ ]:
# veo los 10 mejores hiperparámetros
tb_grid_search[1:10]

cp,maxdepth,minsplit,minbucket,ganancia_mean,qty
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>
-1,8,1000,150,494966667,5
-1,8,1000,75,484200000,5
-1,8,1000,100,484083333,5
-1,8,1000,125,482633333,5
NA,NA,NA,NA,NA,NA
NA,NA,NA,NA,NA,NA
NA,NA,NA,NA,NA,NA
NA,NA,NA,NA,NA,NA
NA,NA,NA,NA,NA,NA


In [ ]:
# genero un id a la tabla
tb_grid_search[, id := .I ]

fwrite( tb_grid_search,
  file = "gridsearch5.txt",
  sep = "\t"
)


# 4.  Análisis de resultados de Grid Search

La salida de la corrida anterior queda en ~/buckets/b1/exp/HT2900  que corresponde a su Google Drive
<br>HT significa Hyperparameter Tuning
<br>El Grid Search es un método de fuerza bruta de un altísimo costo computacional.
<br>Queremos ver si es posible crear un algoritmo de optimización de hiperparámetros que se ahorre recorrer ciertas porciones muy malas del espacio de búsqueda. Algo del estilo “cada vez que pruebo una combinación de hiperparámetros donde  cp > 1 , la ganancia es muy mala, con lo cual ni vale la pena perder el tiempo explorando en esa region”


<br>Levante el archivo de salida gridsearch.txt  a una planilla tipo Excel y analícelo detenidamente
<br>Ordene por ganancia_mean descendente
<br>
<br>En la Planilla Colaborativa, Hoja  C3-GridSearch  cargue el mejor del ranking en la posición 1, el segundo en la 2, y el 5, 10, 50 y 100. Es decir debe cargr SEIS lineas en las celdas correspondientes a su nombre.
<br>
<br>Verifique que efectivamente está dado de alta en la competencia Kaggle  "Data Mining, Inicial 2026 B"  lo que debio haber hecho siguiendo el capítulo Arranque en Frio de El Libro de la Asignatura
<br>En la Planilla Colaborativa, Hoja  C3-GridSearch, utilizando el notebook  **src/arboles/z102_FinalTrain.ipynb**  haga el submit a Kaggle de cada una de las SEIS combinaciones de hiperparámetros y completa la columna Public Leaderboard

<br>
<br>El de mayor ganancia_mean  decimos que es el primero del ranking
En Zulip, correspondiente channel  #Tarea Hogar 02 , topic Analisis Grid Search   intente contestar estas preguntas:

* ¿Qué combinaciones de hiperparámetros poseen una ganancia muy buena?
* ¿Hay algun hiperparámetro que para cierto valor siempre genera una ganancia muy mala, a independientemente de lo que valgan los otros hiperparámetros ?
* ¿Que combinaciones de hiperparámetros es pésima y hubiera sido bueno ahorrarse esas corridas ?

( tiempo estimado 40 minutos, dificultad media )

¿Qué combinaciones de hiperparámetros poseen una ganancia muy buena?

* Las mejores corridas comparten cp=-1 y maxdepth bajo (6 u 8) con minsplit flexible (entre 20 y 1000)

¿Hay algun hiperparámetro que para cierto valor siempre genera una ganancia muy mala, a independientemente de lo que valgan los otros hiperparámetros ?

No hay un solo hiperparámetro que por sí solo garantice mala ganancia en todas las combinaciones. Pero se encontraron las siguientes interacciones:

* Cuando cp = 0 y minbucket = 100, la ganancia es siempre 0, sin importar qué valores tomen maxdepth o minsplit


¿Que combinaciones de hiperparámetros es pésima y hubiera sido bueno ahorrarse esas corridas ?

Las corridas con ganancia = 0 . La combinación encontrada :

* Cualquier combinación con cp = 0 y minbucket = 100, siempre da 0, sin excepción.

* cp = 0 y maxdepth = 4 siempre 0, sin importar minsplit/minbucket.